In [2]:
from pathlib import Path

import polars as pl

from paretodo.data_models import (
    InvestigationSpace,
    InvestigationSpaceVariable,
    OptimizationAxis,
    OptimizationProblem
)
from helpers.sampling.sampling_lhs import lhs_sampling
from helpers.problems.hb_butterot_simulator import HimmelblauButterotSimulator


In [3]:
LOCAL_DIR = "../.local/himmelblau/"
Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)

In [4]:
investigation_space = InvestigationSpace(
    variables=[
        InvestigationSpaceVariable(
            name="x1",
            min=-5.5,
            max=5.5,
            discretization=0.01
        ),
        InvestigationSpaceVariable(
            name="x2",
            min=-5.5,
            max=5.5,
            discretization=0.01
        )
    ]
)

investigation_space.to_json(Path(f"{LOCAL_DIR}/is_llm.json"))

In [8]:
optimisation_problem = OptimizationProblem(
    problem=[
        OptimizationAxis(
            var_name="product_concentration",
            minimize=False,
            optimization_type="objective",
            lower_bound=None,
            upper_bound=None,
        ),
        OptimizationAxis(
            var_name="product_concentration",
            minimize=False,
            optimization_type="standard_deviation",
            lower_bound=None,
            upper_bound=None,
        ),
    ]
)

optimisation_problem.to_json(Path(f"{LOCAL_DIR}/op_llm.json"))

In [9]:
N_SAMPLES = 20000
SEED = 11

lhs_samples = lhs_sampling(
    investigation_space=investigation_space,
    n_samples=N_SAMPLES,
    seed=SEED
)

simulator = HimmelblauButterotSimulator()
experiment_dataset = simulator.run_simulation(lhs_samples, iteration=0)

df = experiment_dataset.to_dataframe()
df.write_csv(f"{LOCAL_DIR}/testset_full.csv")

df_filtered = df.filter(pl.col("product_concentration") > 2000)
df_filtered.write_csv(f"{LOCAL_DIR}/testset_objective.csv")
